<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/stage_07_03_gru_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_03 - GRU (Seq2Seq)**


## **Introducción**

El GRU (Gated Recurrent Unit) es una red neuronal recurrente diseñada para modelar secuencias temporales, manteniendo memoria del pasado de forma explícita.

A diferencia del MLP:

- el GRU no aplana la ventana de entrada,  
- procesa la secuencia paso a paso en el tiempo,  
- aprende dependencias temporales entre observaciones consecutivas.

---

**Idea central del GRU**

El GRU incorpora compuertas (gates) que le permiten:

- decidir qué información recordar,  
- qué información olvidar,  
- y cómo actualizar su estado interno.

Gracias a este mecanismo, el GRU puede capturar patrones temporales relevantes sin incurrir en el costo computacional completo de arquitecturas más complejas como LSTM.

---

**GRU como modelo Seq2Seq en este proyecto**

En un esquema Seq2Seq:

- el Encoder recibe la secuencia de entrada (ventana histórica),  
- el Decoder genera la secuencia de salida (deltas futuros).

En el contexto del proyecto MNQ:

- la entrada es una ventana intradía de longitud fija,  
- la salida es una secuencia multi-step para un horizonte determinado (H=60 o H=90),  
- se entrena un modelo independiente para cada horizonte.

---

**Interpretación según Janse**

El GRU es un modelo clásico y robusto para series temporales que:

- introduce estructura temporal explícita en el modelado,  
- reduce el riesgo de sobreajuste frente a modelos densos sin memoria,  
- representa el siguiente escalón natural después del MLP en complejidad y capacidad predictiva.

## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

In [2]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:

#VENTANAS 60MIN
IN_WINDOW_TRAIN_60_Z = Path(os.environ.get("IN_WINDOW_TRAIN_60_Z", "data/windows/scaled/windows_train_60_z.npz"))
IN_WINDOW_VALID_60_Z = Path(os.environ.get("IN_WINDOW_VALID_60_Z", "data/windows/scaled/windows_valid_60_z.npz"))
IN_WINDOW_TEST_60_Z = Path(os.environ.get("IN_WINDOW_TEST_60_Z", "data/windows/scaled/windows_test_60_z.npz"))
IN_SCALER_60 = Path(os.environ.get("IN_SCALER_60", "data/windows/scaled/scaler_60.pkl"))

#VENTANAS 90MIN
IN_WINDOW_TRAIN_90_Z = Path(os.environ.get("IN_WINDOW_TRAIN_90_Z", "data/windows/scaled/windows_train_90_z.npz"))
IN_WINDOW_VALID_90_Z = Path(os.environ.get("IN_WINDOW_VALID_90_Z", "data/windows/scaled/windows_valid_90_z.npz"))
IN_WINDOW_TEST_90_Z = Path(os.environ.get("IN_WINDOW_TEST_90_Z", "data/windows/scaled/windows_test_90_z.npz"))
IN_SCALER_90 = Path(os.environ.get("IN_SCALER_90", "data/windows/scaled/scaler_90.pkl"))

#ARTIFACTS

# Summary del stage_03a (donde está delta_target_p70 por horizonte).
IN_TARGET_INVESTIGATION_SUMMARY = Path(os.environ.get("IN_TARGET_INVESTIGATION_SUMMARY", "reports/stage_03a_target_investigation_summary.json"))

# Summary del stage_06 (donde está window_size y n_features por horizonte).
IN_WINDOWS_SCALING_SUMMARY =Path(os.environ.get("IN_WINDOWS_SCALING_SUMMARY", "reports/stage_06_window_scaling_seq2seq_summary.json"))

#OUT_MODEL_METRICS = Path(os.environ.get("OUT_MODEL_METRICS", f"reports/stage_07__model_metrics.json"))

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


In [4]:
#PARA LA NOTEBOOK
IN_WINDOW_TRAIN_60_Z = DRIVE_DIR / IN_WINDOW_TRAIN_60_Z
IN_WINDOW_VALID_60_Z = DRIVE_DIR / IN_WINDOW_VALID_60_Z
IN_WINDOW_TEST_60_Z = DRIVE_DIR / IN_WINDOW_TEST_60_Z
IN_SCALER_60 = DRIVE_DIR / IN_SCALER_60

IN_WINDOW_TRAIN_90_Z = DRIVE_DIR / IN_WINDOW_TRAIN_90_Z
IN_WINDOW_VALID_90_Z=DRIVE_DIR / IN_WINDOW_VALID_90_Z
IN_WINDOW_TEST_90_Z = DRIVE_DIR / IN_WINDOW_TEST_90_Z
IN_SCALER_90 = DRIVE_DIR / IN_SCALER_90

IN_WINDOWS_SCALING_SUMMARY = DRIVE_DIR / IN_WINDOWS_SCALING_SUMMARY
IN_TARGET_INVESTIGATION_SUMMARY = DRIVE_DIR / IN_TARGET_INVESTIGATION_SUMMARY

## **2. Reproducibilidad**

In [5]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **3. Configuración**

In [6]:
def _read_json(path: Path) -> Dict[str, Any]:
    """Lee un JSON y devuelve un dict Python (con validación básica de existencia)."""
    # Verifica que el archivo exista antes de abrirlo.
    if not path.exists():
        # Si no existe, corta la ejecución con un error claro.
        raise FileNotFoundError(f"No existe el JSON: {path}")
    # Abre el archivo en modo lectura, asegurando UTF-8.
    with path.open("r", encoding="utf-8") as f:
        # Parsea el contenido JSON y lo devuelve como dict.
        return json.load(f)

In [7]:
# Config final para entrenar (modelo, horizonte).
@dataclass(frozen=True)
class StageConfig:
    # Horizonte (60 o 90).
    horizon: int
    # Largo de ventana (timesteps) desde stage_06.
    seq_len: int
    # Cantidad de features desde stage_06 para ese horizonte.
    n_features: int
    # Nombres de features (orden exacto) para ese horizonte.
    feature_names: List[str]
    # Nombre del target para ese horizonte.
    target_name: List[str]
    # Umbral mínimo económico (DELTA_BASE).
    delta_base: float
    # Umbral de oportunidad (DELTA_OP) leído del stage_03a.
    delta_op: float

In [8]:
# Construye un StageConfig leyendo ambos reports.
def load_state_from_reports(
    horizon: int,
    #model_name: str,
    *,
    in_windows_scaling_summary: Path = IN_WINDOWS_SCALING_SUMMARY,
    in_target_investigation_summary: Path = IN_TARGET_INVESTIGATION_SUMMARY,
) -> StageConfig:
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Carga JSON del stage_06.
    w = _read_json(in_windows_scaling_summary)

    # Carga JSON del stage_03a.
    t = _read_json(in_target_investigation_summary)

    # Lee window_size global (SEQ_LEN).
    seq_len = int(w["details"]["config"]["window_size"])

    # Selecciona dataset del horizonte (ojo: "60" o "90" como string).
    ds = w["details"]["datasets"][str(horizon)]

    # Lee n_features del horizonte.
    n_features = int(ds["n_features"])

    # Lee feature_names del horizonte (aquí se refleja el 1 feature distinto).
    feature_names = list(ds["feature_names"])

    # Lee target del horizonte.
    target_name = ds["target"]

    # Valida consistencia.
    if len(feature_names) != n_features:
        raise ValueError("Inconsistencia entre n_features y feature_names")

    # Define la key de delta_base_med del stage_03a.
    target_base = f"h{horizon}_delta_base_med"
    delta_base = float(t["metrics"][target_base])

    # Define la key de delta_target_p70 del stage_03a.
    target_op = f"h{horizon}_delta_target_p70"
    # Lee DELTA_OP para ese horizonte.
    delta_op = float(t["metrics"][target_op])

    # Devuelve la config lista para entrenar.
    return StageConfig(
        horizon=horizon,
        seq_len=seq_len,
        n_features=n_features,
        feature_names=feature_names,
        target_name=target_name,
        delta_base=float(delta_base),
        delta_op=float(delta_op),
            )


In [9]:
states_h60 = load_state_from_reports(horizon=60)
states_h60

StageConfig(horizon=60, seq_len=29, n_features=7, feature_names=['open', 'high', 'low', 'close', 'ema_60', 'mom_10_struct', 'roc_60'], target_name='delta_pts_60', delta_base=52.12, delta_op=84.18)

In [10]:
states_h90 = load_state_from_reports(horizon=90)
states_h90

StageConfig(horizon=90, seq_len=29, n_features=7, feature_names=['open', 'high', 'low', 'close', 'ema_60', 'mom_10_struct', 'roc_30'], target_name='delta_pts_90', delta_base=60.75, delta_op=97.22)


## **4. Importar métricas comunes desde .py**

In [11]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2seq_metrics import compute_seq2seq_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [12]:
print(compute_seq2seq_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2seq.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples, seq_len) o (n_samples, seq_len, 1)
    y_pred : np.ndarray
        Valores predichos con shape (n_samples, seq_len) o (n_samples, seq_len, 1)
    compute_r2 : bool
        Si True, calcula R² sobre la secuencia completa concatenada.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 (y_true_last==0 o y_pred_last==0)
        al calcular DA_last. Esto evita ambigüedad en la dirección.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales y diagnóstico por paso.
    


## **5. Carga de data windows**

In [13]:
# --------------------------------------------------
# Función común: carga .npz estándar (X, y)
# --------------------------------------------------
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.

    Espera claves:
    - 'X': array (n_samples, seq_len, n_features)
    - 'y' o 'Y': array (n_samples, seq_len) o (n_samples, seq_len, 1)
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Carga el NPZ (lectura).
    data = np.load(path)

    # Lee X (obligatoria).
    if "X" not in data:
        raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
    X = data["X"]

    # Lee y: soporta 'y' (convención usada) o 'Y' (por compatibilidad).
    if "y" in data:
        y = data["y"]
    elif "Y" in data:
        y = data["Y"]
    else:
        raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Devuelve X e y.
    return X, y

In [14]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [15]:
# --------------------------------------------------
# Carga completa: train/valid/test + scaler por horizonte
# --------------------------------------------------
def load_windows_and_scaler_for_horizon(horizon: int) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler para un horizonte dado (60 o 90).

    Retorna un dict:
    {
      "horizon": 60,
      "paths": {...},
      "scaler": <StandardScaler>,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }
    """
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Selecciona paths según horizonte.
    if horizon == 60:
        train_path = IN_WINDOW_TRAIN_60_Z
        valid_path = IN_WINDOW_VALID_60_Z
        test_path  = IN_WINDOW_TEST_60_Z
        scaler_path = IN_SCALER_60
    else:
        train_path = IN_WINDOW_TRAIN_90_Z
        valid_path = IN_WINDOW_VALID_90_Z
        test_path  = IN_WINDOW_TEST_90_Z
        scaler_path = IN_SCALER_90

    # Carga ventanas.
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    # Carga scaler.
    scaler = load_scaler(scaler_path)

    # Retorna todo empaquetado.
    return {
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [16]:
# --------------------------------------------------
# Carga efectiva: H60 y H90 (dos datasets distintos)
# --------------------------------------------------

# Carga todo para 60 min.
bundle_60 = load_windows_and_scaler_for_horizon(60)

# Carga todo para 90 min.
bundle_90 = load_windows_and_scaler_for_horizon(90)


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [17]:
# --------------------------------------------------
# Verificación rápida
# --------------------------------------------------

# Shapes H60.
print("H60 Train:", bundle_60["train"]["X"].shape, bundle_60["train"]["y"].shape)
print("H60 Valid:", bundle_60["valid"]["X"].shape, bundle_60["valid"]["y"].shape)
print("H60 Test :", bundle_60["test"]["X"].shape,  bundle_60["test"]["y"].shape)

# Shapes H90.
print("H90 Train:", bundle_90["train"]["X"].shape, bundle_90["train"]["y"].shape)
print("H90 Valid:", bundle_90["valid"]["X"].shape, bundle_90["valid"]["y"].shape)
print("H90 Test :", bundle_90["test"]["X"].shape,  bundle_90["test"]["y"].shape)

# Información útil (scaler).
print("Scaler H60:", type(bundle_60["scaler"]).__name__)
print("Scaler H90:", type(bundle_90["scaler"]).__name__)

H60 Train: (912, 29, 7) (912, 29)
H60 Valid: (195, 29, 7) (195, 29)
H60 Test : (196, 29, 7) (196, 29)
H90 Train: (912, 29, 7) (912, 29)
H90 Valid: (195, 29, 7) (195, 29)
H90 Test : (196, 29, 7) (196, 29)
Scaler H60: StandardScaler
Scaler H90: StandardScaler


NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **6. Sanity Check**

In [18]:
def sanity_check(
    X: np.ndarray,                 # Tensores de entrada: (n_samples, seq_len, n_features)
    y: np.ndarray,                 # Targets: (n_samples, seq_len) o (n_samples, seq_len, 1)
    name: str,                     # Nombre lógico del split (ej: "train_h60", "valid_h90")
    *,
    expected_seq_len: int,         # Largo de secuencia esperado (ej: 29)
    expected_n_features: int,      # Número de features esperado (ej: 8)
) -> None:
    # --------------------------------------------------
    # Chequeos de dimensionalidad
    # --------------------------------------------------

    # X debe ser estrictamente 3D: (muestras, tiempo, features)
    assert X.ndim == 3, (
        f"{name}: X debe ser 3D (n, seq, feat)"
    )

    # y puede ser 2D (n, seq) o 3D (n, seq, 1)
    assert y.ndim in (2, 3), (
        f"{name}: y debe ser 2D o 3D (n, seq) o (n, seq, 1)"
    )

    # --------------------------------------------------
    # Chequeos de consistencia temporal y estructural
    # --------------------------------------------------

    # Verifica que el largo temporal de X coincida con el esperado
    assert X.shape[1] == expected_seq_len, (
        f"{name}: seq_len inesperado en X: {X.shape[1]} != {expected_seq_len}"
    )

    # Verifica que la cantidad de features en X sea la esperada
    assert X.shape[2] == expected_n_features, (
        f"{name}: n_features inesperado en X: {X.shape[2]} != {expected_n_features}"
    )

    # Verifica que y tenga el mismo largo temporal que X
    assert y.shape[1] == expected_seq_len, (
        f"{name}: seq_len inesperado en y: {y.shape[1]} != {expected_seq_len}"
    )

    # --------------------------------------------------
    # Chequeos numéricos (sanidad de valores)
    # --------------------------------------------------

    # Asegura que X no contenga NaN ni infinitos
    assert np.isfinite(X).all(), (
        f"{name}: X contiene NaN/inf"
    )

    # Asegura que y no contenga NaN ni infinitos
    assert np.isfinite(y).all(), (
        f"{name}: y contiene NaN/inf"
    )

In [19]:
from __future__ import annotations

from typing import Any, Dict
import numpy as np


def run_sanity_checks_for_bundle(bundle: Dict[str, Any], *, tag: str) -> None:
    """
    Ejecuta sanity_check para train/valid/test usando únicamente variables locales.

    Espera un bundle con estructura:
    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
      ...
    }
    """
    # Extrae arrays localmente (no crea globals)
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]

    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]

    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    # Toma expected_* desde TRAIN (consistencia)
    expected_seq_len = int(X_tr.shape[1])
    expected_n_features = int(X_tr.shape[2])

    # Ejecuta sanity checks por split
    sanity_check(X_tr, y_tr, f"train_{tag}", expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)
    sanity_check(X_va, y_va, f"valid_{tag}", expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)
    sanity_check(X_te, y_te, f"test_{tag}",  expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)

    # Mensaje de OK por bundle/horizonte
    h = bundle.get("horizon", "NA")
    print(f"OK {tag} (h={h}) | seq_len={expected_seq_len} | n_features={expected_n_features}")

    # Opcional: borra referencias locales explícitamente (no es estrictamente necesario)
    del X_tr, y_tr, X_va, y_va, X_te, y_te


def run_sanity_checks_all_horizons(bundle_60: Dict[str, Any], bundle_90: Dict[str, Any]) -> None:
    """Corre sanity checks para ambos horizontes."""
    run_sanity_checks_for_bundle(bundle_60, tag="h60")
    run_sanity_checks_for_bundle(bundle_90, tag="h90")

In [20]:
run_sanity_checks_all_horizons(bundle_60, bundle_90)

OK h60 (h=60) | seq_len=29 | n_features=7
OK h90 (h=90) | seq_len=29 | n_features=7


## **7. Definición del modelo — placeholder**

In [108]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

En un MLP Direct Multi-step:

- El riesgo principal es el sobreajuste por capacidad
(muchos parámetros, sin estructura temporal explícita).
- Por eso, la regularización no es opcional, es parte del modelo.

La jerarquía correcta es:

- L2 (weight decay) → mecanismo principal
- Early stopping en VALID → control dinámico
- Arquitectura limitada → prevención estructural
- Dropout leve → solo si hace falta

Tal como indica Jansen:
- Modelo flexible que requiere regularización explícita para generalizar.

### **7.1. Definición de modelo con regularización**

In [141]:
# ---------------------------------------------
# GRU Seq2Seq (SIN initial_state) – FIX robusto
# ---------------------------------------------
# Idea:
# - Encoder GRU produce un vector de contexto (batch, units)
# - RepeatVector(horizon_len) lo repite H veces -> (batch, H, units)
# - Decoder GRU consume esa secuencia y produce (batch, H, units)
# - TimeDistributed(Dense(1)) produce (batch, H, 1)
#
# Ventaja:
# - Elimina por completo el uso de initial_state, evitando el error (64,) en GPU.
# ---------------------------------------------

from tensorflow import keras
from tensorflow.keras import layers

def build_gru_seq2seq_repeatvector(
    *,
    window_len: int = 29,
    n_features: int = 7,
    horizon_len: int = 29,
    units: int = 64,            # hidden state limitado (regularización estructural)
    num_layers: int = 1,        # 1–2 capas (por ahora 1)
    dropout: float = 0.10,      # dropout entre capas
    lr: float = 1e-3,
    run_eagerly: bool = True,
) -> keras.Model:
    """
    GRU Seq2Seq (RepeatVector):
    - Input:  X (window_len, n_features)
    - Output: y_hat (horizon_len, 1)
    """
    if num_layers not in (1, 2):
        raise ValueError("num_layers debe ser 1 o 2.")

    # -------------------------
    # INPUT (solo encoder input)
    # -------------------------
    enc_in = keras.Input(shape=(window_len, n_features), name="enc_in")

    # -------------------------
    # ENCODER
    # -------------------------
    x = enc_in

    # Si se usan 2 capas, la primera devuelve secuencia
    if num_layers == 2:
        x = layers.GRU(
            units,
            return_sequences=True,
            dropout=dropout,
            recurrent_dropout=0.0,
            name="enc_gru_1",
        )(x)

    # Encoder final: produce vector contexto (batch, units)
    context = layers.GRU(
        units,
        return_sequences=False,
        dropout=dropout,
        recurrent_dropout=0.0,
        name="enc_gru_final",
    )(x)

    # -------------------------
    # REPEAT VECTOR (crea "input" del decoder)
    # -------------------------
    # (batch, units) -> (batch, horizon_len, units)
    dec_in = layers.RepeatVector(horizon_len, name="repeat_context")(context)

    # -------------------------
    # DECODER
    # -------------------------
    dec_seq = layers.GRU(
        units,
        return_sequences=True,
        dropout=dropout,
        recurrent_dropout=0.0,
        name="dec_gru",
    )(dec_in)

    # -------------------------
    # OUTPUT
    # -------------------------
    y_hat = layers.TimeDistributed(
        layers.Dense(1, activation="linear"),
        name="y_hat",
    )(dec_seq)

    model = keras.Model(inputs=enc_in, outputs=y_hat, name="GRU_Seq2Seq_RepeatVector")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=[keras.metrics.MAE],
        run_eagerly=run_eagerly,
    )
    return model


In [142]:
# ---------------------------------------------
# 1) Limpiar sesión para evitar quedarnos con modelos viejos
# ---------------------------------------------
from tensorflow.keras import backend as K
K.clear_session()

### **7.2. Construcción de modelo desde bundles**

In [143]:
# ---------------------------------------------
# build_gru_model_for_bundle (para RepeatVector)
# ---------------------------------------------

def build_gru_model_for_bundle(bundle_h: dict):
    # X/y de train para inferir shapes
    X_train = bundle_h["train"]["X"]  # (N, 29, 7)
    y_train = bundle_h["train"]["y"]  # (N, 29) o (N, 29, 1)

    window_len = int(X_train.shape[1])
    n_features = int(X_train.shape[2])

    # y a 3D para inferir horizon_len
    y_train_3d = to_3d_y(y_train)     # (N, H, 1)
    horizon_len = int(y_train_3d.shape[1])

    # Construcción del modelo (sin initial_state)
    model = build_gru_seq2seq_repeatvector(
        window_len=window_len,
        n_features=n_features,
        horizon_len=horizon_len,
        units=64,
        num_layers=1,
        dropout=0.10,
        lr=1e-3,
        run_eagerly=True,
    )
    return model



### **7.3. Helpers**

In [151]:
# ---------------------------------------------
# Helpers para GRU RepeatVector (solo X -> y)
# ---------------------------------------------

def get_gru_xy_from_bundle(bundle_h: dict, *, split: str):
    """
    Devuelve:
      - X:   (N, window_len, n_features)
      - y_3d:(N, horizon_len, 1)
    """
    X = bundle_h[split]["X"]
    y = bundle_h[split]["y"]
    y_3d = to_3d_y(y)
    return X, y_3d

In [152]:
import numpy as np

def to_3d_y(y: np.ndarray) -> np.ndarray:
    """
    Normaliza y a 3D para Seq2Seq:
      - (N, H)    -> (N, H, 1)
      - (N, H, 1) -> (N, H, 1)

    Esto es consistente con la salida del GRU Seq2Seq:
      y_hat: (N, H, 1)
    """
    # Caso: y ya es (N, H, 1)
    if y.ndim == 3:
        if y.shape[-1] != 1:
            raise ValueError(f"Se esperaba y con último dim=1, got {y.shape}")
        return y

    # Caso: y viene como (N, H) y lo expandimos a (N, H, 1)
    if y.ndim == 2:
        return y[..., None]

    # Cualquier otra forma no es compatible con este pipeline
    raise ValueError(f"Forma inesperada para y: {y.shape}")


### **7.4. Verificación de shapes**

In [146]:
# ---------------------------------------------
# PASO 1) Verificación de shapes para GRU Seq2Seq
# ---------------------------------------------
# Objetivo:
# - Confirmar que los bundles H60 y H90 tienen shapes consistentes
# - Obtener (window_len, n_features, horizon_len) desde TRAIN
# - Verificar que VALID mantiene los mismos (window_len, n_features, horizon_len)
#
# Nota:
# - Esto NO entrena nada. Solo inspecciona y valida dimensiones.
# ---------------------------------------------

def assert_same(a: int, b: int, *, msg: str) -> None:
    """Helper pequeño para levantar error con mensaje claro."""
    if int(a) != int(b):
        raise ValueError(f"{msg} (train={a} vs valid={b})")


def check_bundle_shapes_for_gru(bundle_h: dict, *, tag: str) -> dict:
    """
    Chequea shapes relevantes para un bundle (por ejemplo H60 o H90).

    Retorna un dict con:
      - window_len
      - n_features
      - horizon_len
      - train_shapes (X,y)
      - valid_shapes (X,y)
    """
    # -------------------------
    # TRAIN: shapes base del modelo
    # -------------------------
    X_train = bundle_h["train"]["X"]   # (N, window_len, n_features)
    y_train = bundle_h["train"]["y"]   # (N, H) o (N, H, 1)

    window_len_train = int(X_train.shape[1])
    n_features_train = int(X_train.shape[2])

    # Normalizamos y_train a 3D para leer horizon_len de forma uniforme
    y_train_3d = to_3d_y(y_train)      # (N, H, 1)
    horizon_len_train = int(y_train_3d.shape[1])

    # -------------------------
    # VALID: debe ser consistente con TRAIN
    # -------------------------
    X_valid = bundle_h["valid"]["X"]
    y_valid = bundle_h["valid"]["y"]

    window_len_valid = int(X_valid.shape[1])
    n_features_valid = int(X_valid.shape[2])

    y_valid_3d = to_3d_y(y_valid)
    horizon_len_valid = int(y_valid_3d.shape[1])

    # -------------------------
    # Validaciones de consistencia
    # -------------------------
    assert_same(window_len_train, window_len_valid, msg=f"[{tag}] window_len inconsistente")
    assert_same(n_features_train, n_features_valid, msg=f"[{tag}] n_features inconsistente")
    assert_same(horizon_len_train, horizon_len_valid, msg=f"[{tag}] horizon_len inconsistente")

    # -------------------------
    # Reporte compacto
    # -------------------------
    info = {
        "tag": tag,
        "window_len": window_len_train,
        "n_features": n_features_train,
        "horizon_len": horizon_len_train,
        "train_shapes": {
            "X": tuple(X_train.shape),
            "y_raw": tuple(y_train.shape),
            "y_3d": tuple(y_train_3d.shape),
        },
        "valid_shapes": {
            "X": tuple(X_valid.shape),
            "y_raw": tuple(y_valid.shape),
            "y_3d": tuple(y_valid_3d.shape),
        },
    }

    return info


# -------------------------
# Ejecutar checks para H60 y H90
# -------------------------
info_60 = check_bundle_shapes_for_gru(bundle_60, tag="H60")
info_90 = check_bundle_shapes_for_gru(bundle_90, tag="H90")

# Imprimir resumen legible
print(
    f"[H60] window_len={info_60['window_len']} | n_features={info_60['n_features']} | horizon_len={info_60['horizon_len']}\n"
    f"      train X={info_60['train_shapes']['X']} y_raw={info_60['train_shapes']['y_raw']} y_3d={info_60['train_shapes']['y_3d']}\n"
    f"      valid X={info_60['valid_shapes']['X']} y_raw={info_60['valid_shapes']['y_raw']} y_3d={info_60['valid_shapes']['y_3d']}\n"
)

print(
    f"[H90] window_len={info_90['window_len']} | n_features={info_90['n_features']} | horizon_len={info_90['horizon_len']}\n"
    f"      train X={info_90['train_shapes']['X']} y_raw={info_90['train_shapes']['y_raw']} y_3d={info_90['train_shapes']['y_3d']}\n"
    f"      valid X={info_90['valid_shapes']['X']} y_raw={info_90['valid_shapes']['y_raw']} y_3d={info_90['valid_shapes']['y_3d']}\n"
)


[H60] window_len=29 | n_features=7 | horizon_len=29
      train X=(912, 29, 7) y_raw=(912, 29) y_3d=(912, 29, 1)
      valid X=(195, 29, 7) y_raw=(195, 29) y_3d=(195, 29, 1)

[H90] window_len=29 | n_features=7 | horizon_len=29
      train X=(912, 29, 7) y_raw=(912, 29) y_3d=(912, 29, 1)
      valid X=(195, 29, 7) y_raw=(195, 29) y_3d=(195, 29, 1)



### **7.5. Construcción de modelo por horizontes**

In [147]:
# ---------------------------------------------
# PASO 2) Construcción del modelo GRU por horizonte (H60 y H90)
# ---------------------------------------------
# Objetivo:
# - Crear 2 modelos independientes: uno para bundle_60 y otro para bundle_90
# - Asegurar que respeten:
#     input encoder: (None, 29, 7)
#     input decoder: (None, 29, 1)
#     output:        (None, 29, 1)
# - Mostrar un resumen (summary) para verificar arquitectura
#
# Nota:
# - Esto NO entrena nada. Solo construye los modelos.
# ---------------------------------------------

# -------------------------
# Construcción modelo para H60
# -------------------------
model_gru_60 = build_gru_model_for_bundle(bundle_60)

# Imprime resumen de arquitectura (verifique inputs/outputs)
print("\n" + "="*80)
print("MODEL SUMMARY - GRU Seq2Seq (H60)")
print("="*80)
model_gru_60.summary()

# -------------------------
# Construcción modelo para H90
# -------------------------
model_gru_90 = build_gru_model_for_bundle(bundle_90)

# Imprime resumen de arquitectura (verifique inputs/outputs)
print("\n" + "="*80)
print("MODEL SUMMARY - GRU Seq2Seq (H90)")
print("="*80)
model_gru_90.summary()

# -------------------------
# Chequeo rápido de shapes del modelo (opcional, pero útil)
# -------------------------
# Entradas esperadas:
# - encoder input: (None, 29, 7)
# - decoder input: (None, 29, 1)
# Salida esperada:
# - (None, 29, 1)
print("\n" + "="*80)
print("SHAPES CHECK")
print("="*80)
print("H60 inputs:", [t.shape for t in model_gru_60.inputs], "output:", model_gru_60.output.shape)
print("H90 inputs:", [t.shape for t in model_gru_90.inputs], "output:", model_gru_90.output.shape)



MODEL SUMMARY - GRU Seq2Seq (H60)


Model: "GRU_Seq2Seq_RepeatVector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ enc_in (InputLayer)             │ (None, 29, 7)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_gru_final (GRU)             │ (None, 64)             │        14,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_context (RepeatVector)   │ (None, 29, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_gru (GRU)                   │ (None, 29, 64)         │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ y_hat (TimeDistributed)         │ (None, 29, 1)          │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,041 (152.50 KB)

 Trainable params: 39,041 (152.50 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ enc_in (InputLayer)             │ (None, 29, 7)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_gru_final (GRU)             │ (None, 64)             │        14,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_context (RepeatVector)   │ (None, 29, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_gru (GRU)                   │ (None, 29, 64)         │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ y_hat (TimeDistributed)         │ (None, 29, 1)          │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,041 (152.50 KB)

 Trainable params: 39,041 (152.50 KB)

 Non-trainable params: 0 (0.00 B)


SHAPES CHECK
H60 inputs: [(None, 29, 7)] output: (None, 29, 1)
H90 inputs: [(None, 29, 7)] output: (None, 29, 1)


In [148]:
# (Opcional) Confirmar que está en eager
print("H60 run_eagerly:", model_gru_60.run_eagerly)
print("H90 run_eagerly:", model_gru_90.run_eagerly)

H60 run_eagerly: True
H90 run_eagerly: True


In [149]:
print("H60 inputs:", [t.shape for t in model_gru_60.inputs], "output:", model_gru_60.output.shape)
print("H90 inputs:", [t.shape for t in model_gru_90.inputs], "output:", model_gru_90.output.shape)

H60 inputs: [(None, 29, 7)] output: (None, 29, 1)
H90 inputs: [(None, 29, 7)] output: (None, 29, 1)


### **7.7. Preparado de Tensores**

In [154]:
# ---------------------------------------------
# PASO 3) Preparar X/y para TRAIN y VALID (sin entrenar)
# ---------------------------------------------
# Objetivo (RepeatVector):
# - Armar exactamente lo que espera model.fit(...) y model.predict(...)
# - Para cada horizonte:
#     X_train:   (N_train, 29, 7)
#     y_train:   (N_train, 29, 1)
#     X_valid:   (N_valid, 29, 7)
#     y_valid:   (N_valid, 29, 1)
# ---------------------------------------------

def get_gru_xy_from_bundle(bundle_h: dict, *, split: str):
    """
    Devuelve X e y (en 3D) para el GRU RepeatVector.
    - X: (N, window_len, n_features)
    - y: (N, horizon_len, 1)
    """
    X = bundle_h[split]["X"]
    y = to_3d_y(bundle_h[split]["y"])
    return X, y

# -------------------------
# H60: TRAIN y VALID
# -------------------------
X_train_60, y_train_60_3d = get_gru_xy_from_bundle(bundle_60, split="train")
X_valid_60, y_valid_60_3d = get_gru_xy_from_bundle(bundle_60, split="valid")

print("\n" + "="*80)
print("H60 - DATA CHECK (GRU RepeatVector)")
print("="*80)
print("X_train:", X_train_60.shape, "y_train:", y_train_60_3d.shape)
print("X_valid:", X_valid_60.shape, "y_valid:", y_valid_60_3d.shape)

# -------------------------
# H90: TRAIN y VALID
# -------------------------
X_train_90, y_train_90_3d = get_gru_xy_from_bundle(bundle_90, split="train")
X_valid_90, y_valid_90_3d = get_gru_xy_from_bundle(bundle_90, split="valid")

print("\n" + "="*80)
print("H90 - DATA CHECK (GRU RepeatVector)")
print("="*80)
print("X_train:", X_train_90.shape, "y_train:", y_train_90_3d.shape)
print("X_valid:", X_valid_90.shape, "y_valid:", y_valid_90_3d.shape)


H60 - DATA CHECK (GRU RepeatVector)
X_train: (912, 29, 7) y_train: (912, 29, 1)
X_valid: (195, 29, 7) y_valid: (195, 29, 1)

H90 - DATA CHECK (GRU RepeatVector)
X_train: (912, 29, 7) y_train: (912, 29, 1)
X_valid: (195, 29, 7) y_valid: (195, 29, 1)


### **7.8. Entrenamiento GRU**

In [137]:
# -------------------------
# EarlyStopping (clave)
# -------------------------
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True,
)


In [155]:
# ============================================================
# PASO 4) FIT COMPLETO Y ROBUSTO – GRU Seq2Seq (VALID ONLY)
# ============================================================
# Este bloque:
# - Limpia sesión (evita tensores/modelos viejos)
# - Reconstruye modelos H60 y H90
# - Verifica GPU
# - Verifica shapes críticos antes de entrenar
# - Entrena usando EarlyStopping como regularización principal
# - Usa SOLO TRAIN + VALID (no test)
# ============================================================

# -------------------------
# 0) Limpieza de sesión
# -------------------------
# Evita errores por reutilización de grafos/tensores antiguos
K.clear_session()


# -------------------------
# 1) Reconstrucción de modelos
# -------------------------
# IMPORTANTE:
# - build_gru_model_for_bundle DEBE usar la versión actual de build_gru_seq2seq
# - El modelo debe estar compilado con run_eagerly=True
# Verificación explícita de ejecución eager
print("run_eagerly H60:", model_gru_60.run_eagerly)
print("run_eagerly H90:", model_gru_90.run_eagerly)

# -------------------------
# 2) Verificación de GPU
# -------------------------
gpus = tf.config.list_logical_devices("GPU")
device_name = "/device:GPU:0" if gpus else "/device:CPU:0"
print("Dispositivo a usar:", device_name)


# -------------------------
# 3) Verificaciones de SHAPES (RepeatVector)
# -------------------------
def check_xy_shapes(X_train, y_train, X_valid, y_valid, tag: str):
    """
    Verifica consistencia básica de shapes para:
      fit(X_train, y_train) y validation_data=(X_valid, y_valid)
    """
    # X debe ser 3D: (N, window_len, n_features)
    assert X_train.ndim == 3, f"[{tag}] X_train no es 3D"
    assert X_valid.ndim == 3, f"[{tag}] X_valid no es 3D"

    # y debe ser 3D: (N, horizon_len, 1)
    assert y_train.ndim == 3, f"[{tag}] y_train no es 3D"
    assert y_valid.ndim == 3, f"[{tag}] y_valid no es 3D"
    assert y_train.shape[-1] == 1, f"[{tag}] y_train último dim != 1"
    assert y_valid.shape[-1] == 1, f"[{tag}] y_valid último dim != 1"

    # N consistente
    assert X_train.shape[0] == y_train.shape[0], f"[{tag}] N_train inconsistente"
    assert X_valid.shape[0] == y_valid.shape[0], f"[{tag}] N_valid inconsistente"

    # horizon_len consistente entre train y valid
    assert y_train.shape[1] == y_valid.shape[1], f"[{tag}] horizon_len train/valid inconsistente"

    print(f"[{tag}] Shapes OK:",
          "X_train", X_train.shape, "y_train", y_train.shape,
          "| X_valid", X_valid.shape, "y_valid", y_valid.shape)

check_xy_shapes(X_train_60, y_train_60_3d, X_valid_60, y_valid_60_3d, tag="H60")
check_xy_shapes(X_train_90, y_train_90_3d, X_valid_90, y_valid_90_3d, tag="H90")


# -------------------------
# 4) EarlyStopping (regularización principal)
# -------------------------
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True,
)


# -------------------------
# 5) Entrenamiento en GPU (VALID ONLY)
# -------------------------
with tf.device(device_name):

    print("\n" + "=" * 80)
    print("FIT – GRU RepeatVector H60 (VALID ONLY)")
    print("=" * 80)

    hist_60 = model_gru_60.fit(
        X_train_60,
        y_train_60_3d,
        validation_data=(X_valid_60, y_valid_60_3d),
        epochs=200,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1,
    )

    print("\n" + "=" * 80)
    print("FIT – GRU RepeatVector H90 (VALID ONLY)")
    print("=" * 80)

    hist_90 = model_gru_90.fit(
        X_train_90,
        y_train_90_3d,
        validation_data=(X_valid_90, y_valid_90_3d),
        epochs=200,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1,
    )

    #2/3min

run_eagerly H60: True
run_eagerly H90: True
Dispositivo a usar: /device:GPU:0
[H60] Shapes OK: X_train (912, 29, 7) y_train (912, 29, 1) | X_valid (195, 29, 7) y_valid (195, 29, 1)
[H90] Shapes OK: X_train (912, 29, 7) y_train (912, 29, 1) | X_valid (195, 29, 7) y_valid (195, 29, 1)

FIT – GRU RepeatVector H60 (VALID ONLY)
Epoch 1/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 206ms/step - loss: 2675.1108 - mean_absolute_error: 36.2818 - val_loss: 2500.0679 - val_mean_absolute_error: 36.3631
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 196ms/step - loss: 2876.8672 - mean_absolute_error: 36.6732 - val_loss: 2488.5071 - val_mean_absolute_error: 36.2942
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 195ms/step - loss: 2812.2935 - mean_absolute_error: 37.1623 - val_loss: 2475.3655 - val_mean_absolute_error: 36.2177
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 199ms/step - loss: 2973.5410 - mean_absolute_error: 37.5456 - val_loss: 2459.8250 - val_mean_absolute_error: 36.1318
Epoch 5/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 195ms/s

### **8.4. Predicción y evaluación (VALID ONLY)**

In [156]:
# ============================================================
# PREDICCIÓN + EVALUACIÓN (VALID ONLY) – GRU RepeatVector
# ============================================================
# Objetivo:
# - Predecir sobre VALID para H60 y H90
# - Convertir y_true/y_pred a 2D (N, H) para usar compute_seq2seq_metrics
# - Calcular métricas ML (MAE, RMSE, R2, DA_last si aplica)
# ============================================================

import numpy as np
import pandas as pd


# -------------------------
# Helpers de forma (2D / 3D)
# -------------------------
def to_2d_y(y: np.ndarray) -> np.ndarray:
    """
    Normaliza y a 2D:
      - (N, H, 1) -> (N, H)
      - (N, H)    -> (N, H)
    """
    if y.ndim == 3 and y.shape[-1] == 1:
        return y[..., 0]
    if y.ndim == 2:
        return y
    raise ValueError(f"Forma inesperada para y: {y.shape}")


def predict_gru_2d(bundle_h: dict, model, *, split: str = "valid") -> tuple[np.ndarray, np.ndarray]:
    """
    Predicción para GRU RepeatVector (un solo input X).
    Retorna:
      - y_true_2d: (N, H)
      - y_pred_2d: (N, H)
    """
    # X de valid
    X = bundle_h[split]["X"]

    # y verdadero desde el bundle (puede venir 2D o 3D)
    y_true_2d = to_2d_y(bundle_h[split]["y"])

    # Predicción del modelo:
    # - GRU RepeatVector entrega (N, H, 1)
    y_pred_3d = model.predict(X, verbose=0)

    # Convertimos a 2D (N, H) para métricas
    y_pred_2d = to_2d_y(y_pred_3d)

    return y_true_2d, y_pred_2d

In [157]:
def metrics_to_df(metrics: dict, *, model: str, split: str, horizon: int) -> pd.DataFrame:
    """
    Convierte dict de métricas a tabla (1 fila).
    """
    return pd.DataFrame([{
        "model": model,
        "split": split,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA_last": metrics.get("DA_last"),
    }])


### **8.5. Flujo completo para H60 y H90**

In [158]:
# -------------------------
# H60 (VALID ONLY)
# -------------------------
y_true_60, y_pred_60 = predict_gru_2d(bundle_60, model_gru_60, split="valid")
ml_valid_60 = compute_seq2seq_metrics(y_true_60, y_pred_60, compute_r2=True)

In [159]:
# -------------------------
# H90 (VALID ONLY)
# -------------------------
y_true_90, y_pred_90 = predict_gru_2d(bundle_90, model_gru_90, split="valid")
ml_valid_90 = compute_seq2seq_metrics(y_true_90, y_pred_90, compute_r2=True)

## **9. Métricas ML**

In [160]:
# -------------------------
# Tabla final (VALID ONLY)
# -------------------------
df_valid_60 = metrics_to_df(ml_valid_60, model="gru", split="valid", horizon=60)
df_valid_90 = metrics_to_df(ml_valid_90, model="gru", split="valid", horizon=90)

pd.concat([df_valid_60, df_valid_90], ignore_index=True)

,model,split,horizon_min,MAE,RMSE,R2,DA_last
0,gru,valid,60,32.936343,45.353919,0.181511,0.484536
1,gru,valid,90,52.452826,68.858771,0.057063,0.463918


## **11. Guardar artefactos para Stage_08**

In [161]:
def save_json(obj: Dict[str, Any], path: Path) -> None:
    """Guarda un diccionario como JSON, creando directorios si es necesario."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

In [162]:
from pathlib import Path
from typing import Any, Dict
import numpy as np

def save_evaluation_artifacts(
    *,
    out_dir: Path,
    model_name: str,
    horizon: int,
    ml_valid: Dict[str, Any],
    y_valid: np.ndarray | None = None,
    y_pred_valid: np.ndarray | None = None,
    save_preds: bool = True,
) -> None:
    """
    Guarda SOLO artefactos de VALID (coherente con el esquema del libro):
      - metrics_ml_valid.json
      - pred_valid.npz (opcional)

    No guarda nada de TEST.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Métricas ML (VALID)
    # -------------------------
    save_json(
        {
            "model": model_name,
            "horizon_min": horizon,
            "split": "valid",
            **ml_valid,
        },
        out_dir / "metrics_ml_valid.json",
    )

    # -------------------------
    # Predicciones OOS (VALID) - opcional
    # -------------------------
    if save_preds:
        if y_valid is None or y_pred_valid is None:
            raise ValueError("Si save_preds=True, debe pasar y_valid y y_pred_valid.")
        np.savez_compressed(
            out_dir / "pred_valid.npz",
            y_true=np.asarray(y_valid),
            y_pred=np.asarray(y_pred_valid),
        )

    print(f"OK - artefactos guardados en: {out_dir}")

In [163]:
OUT_DIR_60 = Path("artifacts/03_gru/h60")
OUT_DIR_60 = DRIVE_DIR / OUT_DIR_60 #Solo para la notebook

save_evaluation_artifacts(
    out_dir=OUT_DIR_60,
    model_name="gru_h60",
    horizon=60,
    ml_valid=ml_valid_60,
    y_valid=y_true_60,        # ← 2D seguro
    y_pred_valid=y_pred_60,   # ← 2D
    save_preds=False,
)

OK - artefactos guardados en: /content/drive/MyDrive/neural_profit/artifacts/03_gru/h60


In [164]:
OUT_DIR_90 = Path("artifacts/03_gru/h90")
OUT_DIR_90 = DRIVE_DIR / OUT_DIR_90 #Solo para la notebook

save_evaluation_artifacts(
    out_dir=OUT_DIR_90,
    model_name="gru_h90",
    horizon=90,
    ml_valid=ml_valid_90,
    y_valid=y_true_90,
    y_pred_valid=y_pred_90,
    save_preds=False,
)

OK - artefactos guardados en: /content/drive/MyDrive/neural_profit/artifacts/03_gru/h90


## **12. Resumen**

- **Métricas ML (test):** MAE, RMSE, DA_last, R2
- **Métricas económicas como filtro (test):** Precision, Opportunity_Recall, Coverage
- Artefactos guardados en `reports/stage_07/h{H}/<model_name>/`

- El GRU mejora al Naive, pero no supera al MLP, especialmente en H=60.

- En H=60 capta señal moderada (R² ≈ 0.18), pero queda por debajo del MLP (R² ≈ 0.37).

- En H=90 la señal es débil (R² ≈ 0.06), consistente con el mayor horizonte.

- DA_last ≈ 0.48 indica que la dirección sigue siendo el principal límite.

Conclusión operativa: en este setup y tamaño de datos, MLP > GRU (RepeatVector); el GRU podría necesitar más capacidad o features para aportar ventaja.